In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so `from tools...` imports work from any notebook subfolder.
_p = Path.cwd().resolve()
for _parent in [_p, *_p.parents]:
    if (_parent / 'tools' / 'search_tools.py').exists():
        sys.path.insert(0, str(_parent))
        break
del _p, _parent


In [1]:
%%capture --no-stderr
# %pip install "autogen-agentchat~=0.2.3"

# In Your OAI_CONFIG_LIST file, you must have two configs,
# one with:           "response_format": { "type": "text" }
# and the other with: "response_format": { "type": "json_object" }


[
    {"model": "gpt-4o-mini", "sk-REDACTED": "key go here", "response_format": {"type": "text"}},
]

In [2]:
import autogen
import os
from autogen.agentchat import UserProxyAgent
from autogen.agentchat.assistant_agent import AssistantAgent
from autogen.agentchat.groupchat import GroupChat
os.environ["SERPER_API_KEY"] = "1edefaec0732d11db50b993ba60539510cc55334"
from tools.search_tools import SearchTools




In [3]:
from autogen import ConversableAgent
from autogen import register_function

import os
import json
import requests

def search_internet(query: str) -> str:
        """Useful to search the internet
        about a a given topic and return relevant results"""
        print("Searching the internet...")
        top_result_to_return = 5
        url = "https://google.serper.dev/search"
        payload = json.dumps(
            {"q": query, "num": top_result_to_return, "tbm": "nws"})
        headers = {
            'X-API-KEY': os.environ['SERPER_API_KEY'],
            'content-type': 'application/json'
        }
        response = requests.request("POST", url, headers=headers, data=payload)
        # check if there is an organic key
        if 'organic' not in response.json():
            return "Sorry, I couldn't find anything about that, there could be an error with you serper api key."
        else:
            results = response.json()['organic']
            string = []
            print("Results:", results[:top_result_to_return])
            for result in results[:top_result_to_return]:
                try:
                    # Attempt to extract the date
                    date = result.get('date', 'Date not available')
                    string.append('\n'.join([
                        f"Title: {result['title']}",
                        f"Link: {result['link']}",
                        f"Date: {date}",  # Include the date in the output
                        f"Snippet: {result['snippet']}",
                        "\n-----------------"
                    ]))
                except KeyError:
                    next

            return '\n'.join(string)

        



In [4]:
import asyncio
import autogen
import os
import httpx
from typing import Optional, List, Dict, Tuple, Union
import random  # noqa E402

import matplotlib.pyplot as plt  # noqa E402
import networkx as nx  # noqa E402

import autogen  # noqa E402
from autogen.agentchat.conversable_agent import ConversableAgent  # noqa E402
from autogen.agentchat.assistant_agent import AssistantAgent  # noqa E402
from autogen.agentchat.groupchat import GroupChat  # noqa E402
from autogen.graph_utils import visualize_speaker_transitions_dict 

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk-REDACTED"

# Define a custom HTTP client
class MyHttpClient(httpx.Client):
    def __deepcopy__(self, memo):
        return self
    
llm_config = {
    "config_list": [
        {
            "model": "nemotron",
            "api_type": "ollama",
            "client_host": "https://7r9o21n06von58-11434.proxy.runpod.net/",
        }
    ]
}




def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False


user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin who terminates the chat when the leader agent sends a message with 'TERMINATE' mentioned it it",
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
)

Leader = ConversableAgent(
    name="Leader",
    system_message=(
        "You are Leader.\n"
        "Task: 4 medical specialists (Cardiology=A, Neurology=B, Endocrinology=C, Infectious=D) must diagnose a complex patient.\n"
        "You instruct them, gather their hypotheses, and integrate into one final diagnosis.\n"
        "Wait for the results before concluding"
        "After discussion, finalize diagnosis and say 'TERMINATE'.\n"

    ),
    llm_config=llm_config
)




AgentA = ConversableAgent(
    name="Cardiologist",
    system_message=(
        "You are Cardiologist and have to work on diagnosing a patient.\n"
        "Focus: cardiac causes of chest pain, arrhythmias.\n"
        "Adapt as new tests arrive.\n"
        "Be concise, follow Leader.\n"
    ),
    llm_config=llm_config
)


AgentB = ConversableAgent(
    name="Neurologist",
    system_message=(
        "You are Neurologist and have to work on diagnosing a patient.\n"
        "Focus: dizziness, numbness, lesions.\n"
        "Incorporate new imaging.\n"
        "Be concise, follow Leader.\n"
    ),
    llm_config=llm_config
)


AgentC = ConversableAgent(
    name="Endocrinologist",
    system_message=(
        "You are Endocrinologist and have to work on diagnosing a patient..\n"
        "Focus: hormonal issues, thyroid abnormalities.\n"
        "Adjust with new test results.\n"
        "Be concise, follow Leader.\n"
    ),
    llm_config=llm_config
)

AgentD = ConversableAgent(
    name="Infectious_Specialist",
    system_message=(
        "You are Infectious Specialist and have to work on diagnosing a patient.\n"
        "Focus: infection/inflammation signs.\n"
        "Adjust diagnosis if fever or markers appear.\n"
        "Be concise, follow Leader.\n"
    ),
    llm_config=llm_config
)



AgentE = ConversableAgent(
    name="Result_Provider",
    system_message=(
    "You are going to provide report results for the patient in the middle of a conversation \n"
    "Say these exact statements below and end your message with passing the next speaker to the leader \n"
    "Statement - Blood Tests: Thyroid Panel: Elevated TSH, low T3/T4. Autoimmune Markers: Positive for anti-thyroid peroxidase (anti-TPO) antibodies. Imaging (MRI of Brain): Subtle, non-specific white matter lesions in the frontal lobe. No large tumors or acute infarcts, Mild elevated ESR (Erythrocyte Sedimentation Rate) and CRP .Slight fever recorded: 37.8°C (100.0°F). Cardiac Monitoring (Holter): Occasional premature atrial contractions, but no sustained dangerous arrhythmias." 
    ),
    llm_config=llm_config
)


Agent5 = ConversableAgent(
    name="Tool_executor",
    system_message=(
        "you are responsible for running the tools"
    ),
    llm_config=llm_config,
)




In [5]:
# Define your agents
agents = [AgentA, AgentB, AgentC, AgentD, Leader, user_proxy, Agent5, AgentE]


In [6]:
def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False

group_chat = GroupChat(
    agents=agents,
    messages=[],
    max_round=50,
    # allowed_or_disallowed_speaker_transitions=allowed_speaker_transitions_dict,
    # speaker_transitions_type="allowed",
)
# Create the manager
manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    code_execution_config=False,
)


In [7]:
# chat_result = user_proxy.initiate_chat(Agent0, message="search internet about google. Use production Manager first")

In [8]:
# Prepare the initial message with the novel's text
initial_message = (
    " Agents A-D, patient has chest pain, arrhythmia, dizziness, numbness, fatigue, weight changes, cold sensitivity. "
    "Initial tests show mild arrhythmias, unclear neuro signs, possible thyroid issues. "
    "Work together. Some more results will be provided mid execution"
    "Perform a diagnosis and propose a plan for the patient. "
    "Make the first speaker endocrinologist and the fifth speaker result_provider - this should not be called before 5th turn"
)

# Initiate the conversation
user_proxy.initiate_chat(manager, message=initial_message)

User_proxy (to chat_manager):

 Agents A-D, patient has chest pain, arrhythmia, dizziness, numbness, fatigue, weight changes, cold sensitivity. Initial tests show mild arrhythmias, unclear neuro signs, possible thyroid issues. Work together. Some more results will be provided mid executionPerform a diagnosis and propose a plan for the patient. Make the first speaker endocrinologist and the fifth speaker result_provider - this should not be called before 5th turn

--------------------------------------------------------------------------------

Next speaker: Endocrinologist


>>>>>>>> USING AUTO REPLY...
Endocrinologist (to chat_manager):

**Turn 1: Endocrinologist (You)**

* **Initial Assessment:**
	+ Symptoms: Chest pain, arrhythmia, dizziness, numbness, fatigue, weight changes, cold sensitivity
	+ Initial Tests: Mild arrhythmias, unclear neuro signs, possible thyroid issues
* **Endocrinological Focus: Thyroid Abnormalities**
* **Preliminary Diagnosis:** Potential Hypothyroidism or Th

ChatResult(chat_id=None, chat_history=[{'content': ' Agents A-D, patient has chest pain, arrhythmia, dizziness, numbness, fatigue, weight changes, cold sensitivity. Initial tests show mild arrhythmias, unclear neuro signs, possible thyroid issues. Work together. Some more results will be provided mid executionPerform a diagnosis and propose a plan for the patient. Make the first speaker endocrinologist and the fifth speaker result_provider - this should not be called before 5th turn', 'role': 'assistant', 'name': 'User_proxy'}, {'content': "**Turn 1: Endocrinologist (You)**\n\n* **Initial Assessment:**\n\t+ Symptoms: Chest pain, arrhythmia, dizziness, numbness, fatigue, weight changes, cold sensitivity\n\t+ Initial Tests: Mild arrhythmias, unclear neuro signs, possible thyroid issues\n* **Endocrinological Focus: Thyroid Abnormalities**\n* **Preliminary Diagnosis:** Potential Hypothyroidism or Thyrotoxicosis (given the broad symptom range)\n* **Immediate Next Steps:**\n\t1. **Thyroid Fu

In [14]:
content = """Thank you for providing the comprehensive report results. After reviewing each component, I have integrated these findings with our initial hypotheses from the medical specialists (Cardiology = A, Neurology = B, Endocrinology = C, Infectious = D) to update our diagnosis and treatment plan.

INTEGRATED ANALYSIS:

    Blood Tests:

        Thyroid Panel: Elevated TSH and low T3/T4 support the diagnosis of Hypothyroidism, consistent with Agent C’s (Endocrinology) findings.

        Autoimmune Markers: Positive anti-TPO antibodies point toward Hashimoto's Thyroiditis.

    Imaging (MRI of Brain):

        Subtle White Matter Lesions: These non-specific findings may relate to the patient’s neurological symptoms, as raised by Agent B (Neurology), though the connection remains somewhat uncertain without further data.

    Additional Lab Results:

        Mildly Elevated ESR and CRP: Indicate a chronic inflammatory process, which aligns with autoimmune thyroiditis; however, the evidence is not strong enough to support an acute infectious process, making Agent D’s (Infectious) concerns less likely.

        Slight Fever: This could be a minor sign of inflammation and should be monitored.

    Cardiac Monitoring (Holter):

        Occasional Premature Atrial Contractions: These findings support Agent A’s (Cardiology) observation of mild cardiac involvement, possibly related to the hypothyroid state.

REFINED DIAGNOSIS AND TREATMENT PLAN:

    Primary Diagnosis: Hashimoto's Thyroiditis with Secondary Hypothyroidism

    Associated Considerations:

        Possible Peripheral Neuropathy: Suggested by subtle MRI findings and clinical symptoms.

        Mild Cardiac Involvement: As evidenced by occasional atrial contractions.

UPDATED MULTIDISCIPLINARY TREATMENT PLAN:

    Endocrinology (Agent C):

        Begin Thyroid Hormone Replacement Therapy.

        Schedule follow-ups for thyroid function tests every 6–8 weeks until levels stabilize.

    Neurology (Agent B):

        Arrange additional neurological assessments to evaluate the possibility of peripheral neuropathy.

        Initiate symptomatic treatment if neuropathic symptoms are confirmed.

    Cardiology (Agent A):

        Continue periodic cardiac monitoring with follow-up Holter studies.

        Advise on lifestyle modifications to support cardiac health.

    Infectious Disease (Agent D):

        Maintain supportive care and monitor for any signs of infection, though current data do not strongly indicate an infectious process.

    Interdisciplinary Coordination:

        Plan regular meetings to review treatment progress and adjust management strategies as needed.

NEXT STEPS:

    Patient Education: Inform the patient about the refined diagnosis and treatment plan, emphasizing the importance of adherence.

    Treatment Initiation: Start the thyroid hormone replacement therapy and any additional recommended treatments.

    Follow-Up: Establish a routine follow-up schedule to monitor thyroid levels, inflammatory markers, and cardiac status."""
print(content)

Thank you for providing the comprehensive report results. After reviewing each component, I have integrated these findings with our initial hypotheses from the medical specialists (Cardiology = A, Neurology = B, Endocrinology = C, Infectious = D) to update our diagnosis and treatment plan.

INTEGRATED ANALYSIS:

    Blood Tests:

        Thyroid Panel: Elevated TSH and low T3/T4 support the diagnosis of Hypothyroidism, consistent with Agent C’s (Endocrinology) findings.

        Autoimmune Markers: Positive anti-TPO antibodies point toward Hashimoto's Thyroiditis.

    Imaging (MRI of Brain):

        Subtle White Matter Lesions: These non-specific findings may relate to the patient’s neurological symptoms, as raised by Agent B (Neurology), though the connection remains somewhat uncertain without further data.

    Additional Lab Results:

        Mildly Elevated ESR and CRP: Indicate a chronic inflammatory process, which aligns with autoimmune thyroiditis; however, the evidence is not

In [10]:
def save_conversation_to_file(groupchat, filename="chat.txt"):
    """
    Save the entire conversation history to a specified file.

    Args:
        groupchat (GroupChat): The GroupChat instance containing the messages.
        filename (str): The name of the file to save the conversation history.
    """
    if not groupchat.messages:
        print("No messages in the group chat to save.")
        return

    # Compile the conversation history
    conversation_history = "\n".join(
        f"{msg['role']}: {msg['content']}" for msg in groupchat.messages
    )

    # Write the conversation history to the file
    with open(filename, "w", encoding="utf-8") as file:
        file.write(conversation_history)

    print(f"Conversation history saved to {filename}")
    
save_conversation_to_file(group_chat, filename="chat.txt")


Conversation history saved to chat.txt


In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Initialize the Llama 3.1 model
llm = ChatOpenAI(
    model="llama3.1",
    base_url="http://44.221.48.158:11434/v1"
)

def structure_logs_with_local_llm(file_path, initial_message):
    """
    Reads chat logs from a file and generates a structured summary.

    Args:
        file_path (str): Path to the chat log file.
        initial_message (str): The initial task or prompt for context.

    Returns:
        str: The structured summary generated by the LLM.
    """
    # Read the chat logs from the file
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            log_context = file.read()
    except FileNotFoundError:
        return "Error: The specified file was not found."
    except Exception as e:
        return f"An error occurred while reading the file: {e}"

    # Prepare the messages for the LLM
    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional chat summarizer who goes through the entire chat and creates a proper summary based on the '{initial_message}'."
            )
        },
        {
            "role": "user",
            "content": (
                f"Convert the following agent logs into a structured format and into a proper summarized final output "
                f"based on the task '{initial_message}'.\n\nLogs:\n{log_context}"
            )
        }
    ]

    # Generate the structured summary using the LLM
    try:
        response = llm.invoke(messages)
        if isinstance(response, AIMessage):
            structured_summary = response.content
        else:
            structured_summary = "Unexpected response type from the model."
    except Exception as e:
        structured_summary = f"An error occurred during LLM processing: {e}"

    return structured_summary


In [12]:
# Define the path to your chat log file and the initial task message
file_path = 'chat.txt'
initial_message = 'Design a comprehensive digital marketing course.'

# Generate the structured summary
summary = structure_logs_with_local_llm(file_path, initial_message)

# Output the summary
print(summary)

An error occurred during LLM processing: Request timed out.
